# Title: 0_Assembly
# Goal: Assemble/clean all reads using UGA Reference Genome

# Date Source: 2016 data available via NCBI (PRJNA704280), 2017 data available upon acceptance.

# 0.1 FAST QC
*Goal* Examine quality of raw reads for both datasets (i.e., 2016 and 2017)

/// bash 
#FASTQC Report of all initial reads (for 2016 data only)
#!/bin/bash
#SBATCH -J initial_fastqc2016RAWData
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 6
#SBATCH --mem=120g
#SBATCH -t 5-0:0:0
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o initial_fastqc%j.o
#SBATCH -e initial_fastqc%j.e
#SBATCH --account=rrg-barrett
module load fastqc/0.12.0
fastqc *.fastq -o . /home/mary2018/projects/rrg-barrett/mary2018/2016_Samples_RAW_Data/RawDataQCReports -t 6
sbatch initial_fastqc2016RAWData.sh 

    #Practice script for a single QC report
            #!/bin/bash
            #SBATCH -J practicefastqc
            #SBATCH -N 1
            #SBATCH -n 1
            #SBATCH -c 1
            #SBATCH --mem=40g
            #SBATCH -t 1:0:0
            #SBATCH --mail-type=BEGIN,END,FAIL
            #SBATCH --mail-user=mk.hickox@mail.mcgill.ca
            #SBATCH -o initial_fastqc%j.o
            #SBATCH -e initial_fastqc%j.e
            #SBATCH --account=rrg-barrett
            module load fastqc/0.12.0
            fastqc HI.4271.002.Index_10.LombardiSP16_R1.fastq -o . /home/mary2018/projects/rrg-barrett/mary2018/practice
            sbatch initial_fastqc2016RAWData.sh 

#DOWNLOAD FASTQC-html
sftp mary2018@beluga.alliancecan.ca
get path-of-file-you-want #Note that this causes the html to downlaod on my local computer under mkhickox
/home/mary2018/projects/rrg-barrett/mary2018/2017_Samples_RAW-Data/Sample_Pool_Laguna_Fall_2-2928759/fastqc.R1/Pool_Laguna_Fall_2-2928759_S1_L001_R1_001_fastqc.html


#MultiQC (to get all QC Reports in 1 file)
    #I previously downloaded the package in a virtual env. called "UpdatedENV" (https://docs.alliancecan.ca/wiki/Python#Creating_and_using_a_virtual_environment)
module load python/3.11.2
source UpdatedENV/bin/activate
multiqc . --ignore "*_I*" --filename "Raw2017MultiQCFinal" #The script. For 2017, had to ignore the Index files hence "--ignore"


        #The Script I used to make the virtual env and download the package 
        virtualenv --no-download UpdatedENV
        source UpdatedENV/bin/activate
        pip install --no-index --upgrade pip
        pip install /home/mary2018/multiqc-1.16.dev0-py3-none-any.whl --no-index #MultiQC package

mv multiqc-1.16.dev0-py3-none-any.whl /home/mary2018/projects/rrg-barrett/mary2018/UpdatedENV

#TRIMMING-  TRY FASTP!!! (for 2016 data, ran this on the uncompressed folder, all files in the same folder!)
#!/bin/bash
#SBATCH --mem=16000M
#SBATCH --time=24:00:00
#SBATCH --account=rrg-barrett
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH -o trimming%j.o
#SBATCH -e trimming%j.e

module load fastp/0.23.4

for R1 in *1.fastq.gz  
do
name=`echo $R1 | sed 's/.1.fastq.gz\+//'`       
R2=${R1%1.fastq.gz}2.fastq.gz
fastp -i $R1 -I $R2 -o trimmed${R1} -O trimmed${R2} --unpaired1 unpaired${R1} --unpaired2 unpaired${R2} --failed_out ${name}_failed.fastq.gz --detect_adapter_for_pe -q 20 -l 50 -g
done

#THings different from Alan/Antoine: Didn't use popoolation, cut polyG, forced cutting adapters 
#Mathilde script disabpled length filtering (-L)... don't know why 
#Mathilde also didn't detect adapters or force the trimming of polyG.
#Mathilde did force the reading to be done from 3->5' end (-3). Didn't find anything about this for Popoolation.

#Downloading html fastp output
/home/mary2018/projects/rrg-barrett/mary2018/2016_Samples_RAW_Data/trimmed/fastp.html 

#Retrying filtering with the tail end cutting (for 2016, 2017 only do this way)
#!/bin/bash
#SBATCH --mem=16000M
#SBATCH --time=24:00:00
#SBATCH --account=rrg-barrett
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH -o trimming%j.o
#SBATCH -e trimming%j.e

module load fastp/0.23.4
cd /home/mary2018/projects/rrg-barrett/mary2018/2016_Samples_RAW_Data/UncompressedRawData

for R1 in *1.fastq.gz  
do
name=`echo $R1 | sed 's/.1.fastq.gz\+//'`       
R2=${R1%1.fastq.gz}2.fastq.gz
fastp -i $R1 -I $R2 -o trimmed${R1} -O trimmed${R2} --unpaired1 unpaired${R1} --unpaired2 unpaired${R2} --failed_out ${name}_failed.fastq.gz --detect_adapter_for_pe -q 20 -l 50 -g -3
done

#Copying fastq files in 2017 datafile, to make currect set up for trimming
#!/bin/bash
#SBATCH -J backup
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 4
#SBATCH --mem=9g
#SBATCH -t 2:00:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o  copy%j.o
#SBATCH -e copy%j.e
#SBATCH --account=rrg-barrett

cp /home/mary2018/projects/rrg-barrett/mary2018/2017_Samples_RAW-Data/RawData/*/*R*_*.fastq.gz /home/mary2018/projects/rrg-barrett/mary2018/2017_Samples_RAW-Data/RawData/FinalCopiedFASTQ

#Filtering 2017DATA

#!/bin/bash
#SBATCH --mem=16000M
#SBATCH --time=24:00:00
#SBATCH --account=rrg-barrett
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH -o trimming%j.o
#SBATCH -e trimming%j.e

module load fastp/0.23.4

cd /home/mary2018/projects/rrg-barrett/mary2018/2017_Samples_RAW-Data/RawData/FinalCopiedFASTQ


for R1 in *1_001.fastq.gz  
do
name=`echo $R1 | sed 's/.1_001.fastq.gz\+//'`       
R2=${R1%1_001.fastq.gz}2_001.fastq.gz
fastp -i $R1 -I $R2 -o trimmed${R1} -O trimmed${R2} --unpaired1 unpaired${R1} --unpaired2 unpaired${R2} --failed_out ${name}_failed.fastq.gz --detect_adapter_for_pe -q 20 -l 50 -g -3
done

#FASTQC OF trimmed
    #2016

!/bin/bash
#SBATCH -J trimmed_fastqc2016
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 6
#SBATCH --mem=120g
#SBATCH -t 1-0:0:0
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o trimmed_fastqc2016%j.o
#SBATCH -e trimmed_fastqc2016%j.e
#SBATCH --account=rrg-barrett
module load fastqc/0.11.9 #Had to load the older version because the new one can't read zipped files!!
fastqc trimmed*.fastq.gz  -o . /home/mary2018/projects/rrg-barrett/mary2018/2016_Samples_RAW_Data/TrimmedDataQCReports -t 6

    #2017

!/bin/bash
#SBATCH -J trimmed_fastqc2017
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 6
#SBATCH --mem=120g
#SBATCH -t 1-0:0:0
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o trimmed_fastqc2017%j.o
#SBATCH -e trimmed_fastqc2017%j.e
#SBATCH --account=rrg-barrett
module load fastqc/0.11.9 
fastqc trimmed*.fastq.gz  -t 6


#MultiQc of trimmed reads (2016)
    #Have to run the following script from mary2018 directory (not UpdatedENV)
module load python/3.11.2
source UpdatedENV/bin/activate
multiqc . --filename "Trimmed2017MultiQCFinal" 

/home/mary2018/projects/rrg-barrett/mary2018/2017_Samples_RAW-Data/TrimmedQC/Trimmed2017MultiQCFinal.html        

#Aligning to the genome
#### STEP3: INDEX THE GENOME FOR BWA
#Using UGA v.5 (https://stickleback.genetics.uga.edu/about/) (ALAN + ANTOINE USED BROADS https://useast.ensembl.org/Gasterosteus_aculeatus/Info/Annotation)
#Copy from desktop
scp  /Users/MK_Hickox/Desktop/stickleback_v5.0.1_assembly.fa.gz mary2018@beluga.alliancecan.ca:/home/mary2018/projects/rrg-barrett/mary2018/Reference
gunzip stickleback_v5.0.1_assembly.fa.gz
module load bwa/0.7.17
bwa index stickleback_v5.0.1_assembly.fa

#### STEP4: BWA MEM
#https://bio-bwa.sourceforge.net/bwa.shtml
#General info on alignment and post-alignment steps (https://hbctraining.github.io/In-depth-NGS-Data-Analysis-Course/sessionVI/lessons/01_alignment.html)
 #Updated script with issue in RG fixed!! (per Wing) + changed $name to ${name} for output files and added path (submitting script in own directory rather than moving it later)

 #FINAL  script  
    #for 2016
#!/bin/bash
#SBATCH --time=24:00:00
#SBATCH --account=rrg-barrett
#SBATCH --cpus-per-task=40
#SBATCH --ntasks=1
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=4G
#SBATCH -o finalalign2016%j.o
#SBATCH -e finalalign2016%j.e
#SBATCH --account=rrg-barrett

module load bwa/0.7.17
for R1 in *1.fastq.gz   
do
name=`echo $R1 | sed 's/.1.fastq.gz\+//'`       
R2=${R1%1.fastq.gz}2.fastq.gz
library=`echo $R1 | sed 's/.*\.\([^.]*\)_.*/\1/'`

bwa mem -M -t 40 -R "@RG\tID:${name}\tPL:ILLUMINA\tLB:${library}\tSM:${library}" /home/mary2018/projects/rrg-barrett/mary2018/Reference/stickleback_v5.0.1_assembly.fa $R1 $R2 > ${name}.sam 2> ${name}.BWAmem.log
done

#Things different from Alan/Antoine: -adding SM and LIB info (with newly made "library" variable), added the M (for Picard later), wrote "ILLUMINA" instead of "Illumina"


#2017 script but fixed for the issues wing found (naminng in RG)
#!/bin/bash
#SBATCH --time=40:00:00
#SBATCH --account=rrg-barrett
#SBATCH --cpus-per-task=40
#SBATCH --ntasks=1
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=4G
#SBATCH -o align2017%j.o
#SBATCH -e align2017%j.e
#SBATCH --account=rrg-barrett

module load bwa/0.7.17
for R1 in *1_001.fastq.gz  
do
name=`echo $R1 | sed 's/.1_001.fastq.gz\+//'`     
R2=${R1%1_001.fastq.gz}2_001.fastq.gz  

bwa mem -M -t 40 -R "@RG\tID:${name}\tPL:Illumina\tLB:${name}\tSM:${name}" /home/mary2018/projects/rrg-barrett/mary2018/Reference/stickleback_v5.0.1_assembly.fa $R1 $R2 > ${name}.sam 2> ${name}.BWAmem.log
done

#### STEP5: REMOVE AMBIGUOUS READS FROM THE SAM FILES AND MALE SORTED.BAM FILES
#Generla info https://bioinformatics-core-shared-training.github.io/cruk-summer-school-2017/Day1/Session5-alignedReads.html
#Originally this was all done in one step, but i will break it up so: bam, clean bam, clean-sort bam

#!/bin/bash
#SBATCH --time=34:00:00
#SBATCH --account=rrg-barrett
#SBATCH --cpus-per-task=10
#SBATCH --ntasks=1
#SBATCH --mem=191000M
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH -o bam2016%j.o
#SBATCH -e bam2016%j.e
#SBATCH --account=rrg-barrett

module load samtools/1.17
for i in *.sam  
do  
echo $i  
name=${i%.sam}
echo $name  
samtools view -@ 10 -q 20 -b -S $i > ${name}_2016cleaned.bam
samtools sort ${name}_2016cleaned.bam -o ${name}_sorted_2016.bam
done

#!/bin/bash
#SBATCH --time=8:00:00
#SBATCH --account=rrg-barrett
#SBATCH --cpus-per-task=10
#SBATCH --ntasks=1
#SBATCH --mem=191000M
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH -o bam2017%j.o
#SBATCH -e bam2017%j.e
#SBATCH --account=rrg-barrett

module load samtools/1.17
for i in *.sam  
do  
echo $i  
name=${i%.sam}
echo $name  
samtools view -@ 10 -q 20 -b -S $i > ${name}_2017cleaned.bam
samtools sort ${name}_2016cleaned.bam -o ${name}_sorted_2017.bam
done

#How to check REad Group (RG) info for a bam 
                            samtools view -H trimmedHI.4271.006.Index_6.YoungerFA16_.sam | grep '^@RG'

#Info on merging files (https://ucdavis-bioinformatics-training.github.io/2017-August-Variant-Analysis-Workshop/wednesday/alignment.html)
    #FOR 2016 ONLY
    #Make a directory for each season/estuary combo and then:
#!/bin/bash
#SBATCH --time=02:40:00
#SBATCH --account=rrg-barrett
#SBATCH --cpus-per-task=1
#SBATCH --ntasks=1
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH --mem=400M
#SBATCH -o merge%j.o
#SBATCH -e merge%j.e
#SBATCH --account=rrg-barrett
module load samtools/1.17
samtools merge YNG_SP-2016-merged.bam *sorted_2016.bam

#Additional info:
        https://www.biostars.org/p/9512741/
        https://www.seqanswers.com/forum/bioinformatics/bioinformatics-aa/39825-when-to-merge-data-from-single-library-run-on-multiple-lanes


#Removing PCR Duplicates
                                    ## remove duplicates (a lot of memory required).
                                #Wing suggests running with 20G
                                #!/bin/bash
                                #SBATCH --time=48:00:00
                                #SBATCH --account=rrg-barrett
                                #SBATCH --cpus-per-task=1
                                #SBATCH --ntasks=1
                                #SBATCH --mail-user=mk.hickox@mail.mcgill.ca
                                #SBATCH --mail-type=ALL
                                #SBATCH --mem=190G
                                #SBATCH -o markdup2016%j.o
                                #SBATCH -e markdup2016%j.e
                                #SBATCH --account=rrg-barrett

                                module load java/13.0.2
                                module load picard/2.26.3

                                for i in *.bam
                                do  
                                echo $i  
                                name=${i%.bam}
                                echo $name  

                                java -Xmx150G -XX:-UseGCOverheadLimit -jar $EBROOTPICARD/picard.jar MarkDuplicates -INPUT ${name}.bam -OUTPUT ${name}_dedup.bam -METRICS_FILE ${name}_dup_metrics.txt -REMOVE_DUPLICATES true
                                done

#For my data, I didn't use the loop but submitted each pop separately. This is because I don't know how to make multiple threads with PicardTools and I thought ind. scripts would be faster!

#!/bin/bash
#SBATCH --time=05:00:00
#SBATCH --account=rrg-barrett
#SBATCH --cpus-per-task=1
#SBATCH --ntasks=1
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH --mem=80G
#SBATCH -o markdup2017%j.o
#SBATCH -e markdup2017%j.e
#SBATCH --account=rrg-barrett

module load java/1.8.0
module load picard/2.26.3

java -Xmx70G -XX:-UseGCOverheadLimit -jar $EBROOTPICARD/picard.jar MarkDuplicates -INPUT trimmedPool_Younger_Spring_2-2928770_S12_L001__sorted_2017.bam -OUTPUT YNG_SP-2017-dedup.bam -METRICS_FILE YNG_SP-2017_dup_metrics.txt -REMOVE_DUPLICATES true


#additional info on PCR dup. removal (https://sites.google.com/a/broadinstitute.org/legacy-gatk-forum-discussions/tutorials/6747-how-to-mark-duplicates-with-markduplicates-or-markduplicateswithmatecigar)


#Samtools Flagstat (to get info on the alignment for eah bam file)
#!/bin/bash
#SBATCH --time=02:00:00
#SBATCH --account=rrg-barrett
#SBATCH --cpus-per-task=1
#SBATCH --ntasks=1
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH --mail-type=ALL
#SBATCH --mem=3Mb
#SBATCH -o flagstat2017%j.o
#SBATCH -e flagstat2017%j.e
#SBATCH --account=rrg-barrett

module load samtools/1.17

(for i in *.bam
do 
samtools flagstat $i
done) > out2017.txt

#For both 2016+2017, no duplicates detected. Running with a single file to see if that changes things. If not, might mean that dup. flag (in Picard)
didn;t work 
            module load samtools/1.17
            samtools flagstat Lombardi_FA-2017-dedup.bam


#### CREATE A BAM LIST
    #Copied dedup.bam from 2016+2017 to the same directory
#don't need to submit this as a job, requires basically no memory/time

for f in `ls -S *.bam`
do echo $f >> bam_list.txt
done

#####chmod a+rx bam_list.txt #Changes permissions??? Didn't run this because idk what it is


#### STEP7: SAMTOOLS MPILEUP
#!/bin/bash
#SBATCH -J mpliep3
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=120g
#SBATCH -t 01-12:00:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o attempt3mpilep%j.o
#SBATCH -e attempt3mpilep%j.e
#SBATCH --account=rrg-barrett

module load samtools
samtools mpileup -B -f /home/mary2018/projects/rrg-barrett/mary2018/Reference/stickleback_v5.0.1_assembly.fa -b bam_list.txt > attempt2SCmulti.mpileup

#I am running this again as Attempt 4, because the current mpileup (which I filtered) had the same output file name when 2 identical scripts were running. IDK if this will affect, but running it again to be safe.

#IDENTIFY INDELS##
#!/bin/bash
#SBATCH -J indelid
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=5g
#SBATCH -t 12:00:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o indelid%j.o
#SBATCH -e indelid%j.e
#SBATCH --account=rrg-barrett
perl /home/mary2018/.local/bin/popoolation2_1201/indel_filtering/identify-indel-regions.pl --input finalSCmulti.mpileup --output indel-regions.gtf --indel-window 5 --min-count 1

#when i ran with 120g for 12 hours, finished after 8 hours

#Count the number of lines in the file (i.e. number of indels and filtering)
indel-regions.gtf

 #!/bin/bash
#SBATCH -J linecount
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=1g
#SBATCH -t 1:00:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o linecount%j.o
#SBATCH -e linecount%j.e
#SBATCH --account=rrg-barrett

wc -l SCMulti-MKattempt2Filtered.mpileup > output-line-count2
wc -l finalSCmulti.mpileup > output-line-count 1 #if you make the same output file, they will be overwritten!!


 #FILTER FOR COMPLETNESS##
#!/bin/bash
#SBATCH -J filtermpilep
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=1G
#SBATCH -t 8:00:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o filtermpilep%j.o
#SBATCH -e filtermpilep%j.e
#SBATCH --account=rrg-barrett

grep -vw '[0-5]' attempt2SCmulti.mpileup > SCMulti-filtered.mpileup #Antoine version that has problems!


#This is my first filtering verions but it doesn't work because doesn't specify intergers only

awk '{ for (i = 1; i <= NF; i++) { if (i != 2 && $i < 5) next } } 1' finalSCmulti.mpileup > SCMulti-MyVersionfiltered.mpileup 

#My version which explicitely ignores the second column (position) when searching for matches from 1-5. Without it, I think that positions 1-5 will be removed (rather than just depths of 1-5). I still don't know if the other intergers are also read depths for other populations, but until I hear otherwise, I will assume that we don't want them. I think that this version ignored all lines with entries in column 2?? Wing helped with this incorrect version

#attempt 2 at my modified filtering
awk 'BEGIN {OFS="\t"} {
    flag = 0
    for (i = 1; i <= NF; i++) {
        if (i != 2 && int($i) == $i && $i >= 0 && $i <= 5) {
            flag = 1
            break
        }
    }
    if (flag == 0) {
        print
    }
}' finalSCmulti.mpileup > SCMulti-MKattempt2Filtered.mpileup


#INDEL IDENTIFICATION AND REMOVAL??
perl /home/mary2018/.local/bin/popoolation2_1201/indel_filtering/identify-indel-regions.pl --input SCMulti-MKattempt2Filtered.mpileup --output indel-regionsmkattempt2filtered.gtf --indel-window 5 --min-count 1
#I actually ran this on the unfiltered file. Will try to use the .gtf output on the filtered file after this. Otherwise, I will have to run the id step again before doing the masking.
#running now on my version filtered file (attempt 2)

### STEP9: CREATE A SYNC FILE
#!/bin/bash
#SBATCH -J syncmpileup
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=120g
#SBATCH -t 24:00:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o syncpilep2%j.o
#SBATCH -e syncmpilep2%j.e
#SBATCH --account=rrg-barrett

module load java
java -ea -Xmx2g -jar /home/mary2018/.local/bin/popoolation2_1201/mpileup2sync.jar --input SCMulti-MKattempt2Filtered.mpileup --output 2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync --fastq>

#My attempt 2 filter-> identified indels and now turning into a sync file. Will then mask the indels!

#Filtering the indels
#!/bin/bash
#SBATCH -J maskindels
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=20g
#SBATCH -t 4:00:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o maskindels%j.o
#SBATCH -e maskindels%j.e
#SBATCH --account=rrg-barrett


perl /home/mary2018/.local/bin/popoolation2_1201/indel_filtering/filter-sync-by-gtf.pl --gtf indel-regionsmkattempt2filtered.gtf --input 2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync --output maskedindels-2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync
#I originally ran with 120 g and it finished within 17 minutes

#Removing scaffolds
    #Doing this for the masked version (masked-indels-2hopefully-actual...), and the unmasked versions (2hopefully...)
    #Job failed (Exit code 1) with nothing in error file. Output file is same size as input, so I think "scaffold" isn't in the file. Will try re-running once I know what qunie contents of the first and second columns are

#!/bin/bash
#SBATCH -J remove-scaffolds-masked
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=8G
#SBATCH -t 00:30:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o remove-scaffolds-masked%j.o
#SBATCH -e remove-scaffolds-masked%j.e
#SBATCH --account=rrg-barrett

    grep -v 'scaffold' maskedindels-2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync > masked_chromosomes.sync
    grep 'scaffold' maskedindels-2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync > testmasked_excluded-scaffolds.sync
#This doesn't work because the file doesn't contain the word "scaffolds". Instead, I will be removing the mitochondiral chrom, chrom that is "unplaced" and the Y sex chromosome, and chrom 19

grep -v -E '^(chrY|chrM|chrUn|chrXIX)\s' maskedindels-2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync > masked-autosome-chrom.sync

    #unmasked version
grep -v -E '^(chrY|chrM|chrUn|chrXIX)\s' 2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync > unmasked-autosome-chrom.sync


# Checking scaffold spelling
    #I want to see if the original sync files (without scaffolds removed) contain scaffolds and how they are spelt. Here is a script to check the first column. Could also be the second column, but I don't think so. Note that technically I should have done this for both files before actually attempting to remoev the scaffolds. Instead. I'm running for the unmaksed, as the masked file is being filtered to remove scaffolds.
#!/bin/bash
#SBATCH -J check-scaffolds-original
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=4g
#SBATCH -t 00:30:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o check-scaffolds-maskedc2%j.o
#SBATCH -e check-scaffolds-maskedc2%j.e
#SBATCH --account=rrg-barrett

    awk '{print $2}' 2hopefully-actual-final-javaSCMulti-MKattempt2Filtered.sync | sort -u > searching-chrom-names-unamskedsync-colum2.txt



#Info on chrun: (unplaced contig-Nath 2021 (https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8022941/) narrowed to chrosmose but not into specific gaps)
https://www.biostars.org/p/304113/
https://www.biostars.org/p/417361/
https://www-ncbi-nlm-nih-gov.proxy3.library.mcgill.ca/grc/help/definitions/

#Running FST tests 
#Popoolation2 masked version (indels covered)

#TEST version

#!/bin/bash
#SBATCH -J test-fst
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=1g
#SBATCH -t 0:15:0
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o test-fst%j.o
#SBATCH -e test-fst%j.e
#SBATCH --account=rrg-barrett

perl /home/mary2018/.local/bin/popoolation2_1201/fst-sliding.pl --input masked-autosome-chrom.sync --output masked-autosome-chrom.fst --min-count 2 --min-coverage 5 --max-coverage 100 --min-covered-fraction 0 --window-size 1 --step-size 1 --pool-size 29:38:30:30:28:30:29:28:30:30:29:21:40:40:40:40:40:40:40:40:40:40:40:40 --suppress-noninformative --test


#!/bin/bash
#SBATCH -J fst-popoolation-masked
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=80g
#SBATCH -t 7-0:0:0
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o fst-popoolation-masked%j.o
#SBATCH -e fst-popoolation-masked%j.e
#SBATCH --account=rrg-barrett

perl /home/mary2018/.local/bin/popoolation2_1201/fst-sliding.pl --input masked-autosome-chrom.sync --output REAL-masked-autosome-chrom.fst --min-count 2 --min-coverage 5 --max-coverage 100 --min-covered-fraction 0 --window-size 1 --step-size 1 --pool-size 29:38:30:30:28:30:29:28:30:30:29:21:40:40:40:40:40:40:40:40:40:40:40:40 --suppress-noninformative

#unmasked version

#Ran it with lower g provision, in case I don't need the whole 120g and it will make it go faster??
#Should have changed output file name to UNmasked... but I forgot. As such the output file name will be "masked-autosome-chrom.fst" and I will have to change it.
#!/bin/bash
#SBATCH -J fst-popoolation-unmasked
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=60g
#SBATCH -t 7-0:0:0
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o fst-popoolation-unmasked%j.o
#SBATCH -e fst-popoolation-unmasked%j.e
#SBATCH --account=rrg-barrett

perl /home/mary2018/.local/bin/popoolation2_1201/fst-sliding.pl --input unmasked-autosome-chrom.sync --output masked-autosome-chrom.fst --min-count 2 --min-coverage 5 --max-coverage 100 --min-covered-fraction 0 --window-size 1 --step-size 1 --pool-size 29:38:30:30:28:30:29:28:30:30:29:21:40:40:40:40:40:40:40:40:40:40:40:40 --suppress-noninformative






BAM LIST ORDER:
YNG_FA-2017-dedup.bam
SCOTT_FA-2017-dedup.bam
Lombardi_FA-2017-dedup.bam
OD_SP-2017-dedup.bam
Laguna_Fall-2017-dedup.bam
WAD_FA-2017-dedup.bam
SCOTT_SP-2017-dedup.bam
OD_FA-2017-dedup.bam
Lombardi_SP-2017-dedup.bam
Laguna_SP-2017-dedup.bam
YNG_SP-2017-dedup.bam
WAD_SP-2017-dedup.bam
WAD_SP-2016-dedup.bam
LOMB_SP-2016-dedup.bam
OD_FA-2016-dedup.bam
LOMB_FA-2016-dedup.bam
YNG_FA-2016-dedup.bam
OD_SP-2016-dedup.bam
WAD_FA-2016-dedup.bam
SCOTT_SP-2016-dedup.bam
SCOTT_FA-2016-dedup.bam
LAG_FA-2016-dedup.bam
LAG_SP-2016-dedup.bam
YNG_SP-2016-dedup.bam


                                #Only have within estuary comparisons for 2017 only

                                #!/bin/bash
                                #SBATCH -J 2017-withinestuary-fst
                                #SBATCH -N 1
                                #SBATCH -n 1
                                #SBATCH -c 1
                                #SBATCH --mem=1G
                                #SBATCH -t 00:20:00
                                #SBATCH --mail-type=BEGIN,END,FAIL
                                #SBATCH --mail-user=mk.hickox@mail.mcgill.ca
                                #SBATCH -o 017-withinestuary-fst%j.o
                                #SBATCH -e 017-withinestuary-fst%j.e
                                #SBATCH --account=rrg-barrett

                                awk '{
                                    found = 0;
                                    output = $1 "\t" $2 "\t" $3 "\t" $4 "\t" $5;

                                    for (i = 6; i <= NF; i++) {
                                        if ($i ~ /1:11|2:7|3:9|4:8|5:10|6:12/) {
                                            found = 1;
                                            output = output "\t" $i;
                                        }
                                    }

                                    if (found) {
                                        print output;
                                    }
                                }' REALmasked-autosome-chrom.fst > 2017-withinestuary-REALmasked-autosome-chrom.fst

                                REALmasked-autosome-chrom.fst

                                #Only have within estuary comparisons for 2016 only

                                #!/bin/bash
                                #SBATCH -J 2016-withinestuary-fst
                                #SBATCH -N 1
                                #SBATCH -n 1
                                #SBATCH -c 1
                                #SBATCH --mem=1G
                                #SBATCH -t 00:20:00
                                #SBATCH --mail-type=BEGIN,END,FAIL
                                #SBATCH --mail-user=mk.hickox@mail.mcgill.ca
                                #SBATCH -o 2016-withinestuary-fst%j.o
                                #SBATCH -e 2016-withinestuary-fst%j.e
                                #SBATCH --account=rrg-barrett

                                awk '{
                                    found = 0;
                                    output = $1 "\t" $2 "\t" $3 "\t" $4 "\t" $5;

                                    for (i = 6; i <= NF; i++) {
                                        if ($i ~ /13:19|14:16|15:18|17:24|20:21|22:23/) {
                                            found = 1;
                                            output = output "\t" $i;
                                        }
                                    }

                                    if (found) {
                                        print output;
                                    }
                                }' REALmasked-autosome-chrom.fst > 2016-withinestuary-REALmasked-autosome-chrom.fst


#Withinestuary for ALL YEARS

#!/bin/bash
#SBATCH -J withinestuary-fst-allyears
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH --mem=1G
#SBATCH -t 00:20:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o withinestuary-fst-allyears%j.o
#SBATCH -e withinestuary-fst-allyears%j.e
#SBATCH --account=rrg-barrett

awk '{
    found = 0;
    output = $1 "\t" $2 "\t" $3 "\t" $4 "\t" $5;

    for (i = 6; i <= NF; i++) {
        if ($i ~ /1:11|2:7|3:9|4:8|5:10|6:12|13:19|14:16|15:18|17:24|20:21|22:23/) {
            found = 1;
            output = output "\t" $i;
        }
    }

    if (found) {
        print output;
    }
}' REALmasked-autosome-chrom.fst > AllYearsWithinestuary-REALmasked-autosome-chrom.fst






#FST OUTLIER for Masked Popoolations2 FST (all together) https://medium.com/the-nature-of-food/how-to-run-your-r-script-with-compute-canada-c325c0ab2973
#First, using an interactice job on terminal, download all the packages you need 
module load 
module load StdEnv/2020
module load r/4.2.1
R
install.packages('data.table', repos='https://cloud.r-project.org/')
q() #this is to get out of the interactive job 

library(data.table)
getwd()
setwd("/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/masked/AllYears-WithinEstuary")
FST_all<- fread("AllYearsWithinestuary-REALmasked-autosome-chrom.fst", header = F, data.table = F)
dim(FST_all)
FST_all_info <- FST_all[,1:5]
FST_all <- FST_all[,6:17]
names_all <- sapply(FST_all, function(i)unique(sub('=.*', '', i)))
FST_all <- data.frame(lapply(FST_all, function(i)sub('.*=', '', i)))
names(FST_all) <- names_all
FST_all <- apply(as.matrix(FST_all), 2, as.numeric)
FST_all <- data.frame(t(FST_all))
FST_pairs <- FST_all[,]
FST_pairs <- as.data.frame(FST_pairs)
row.names(FST_pairs)<- c("Younger2017","Scott2017","Lombardi2017","OldDairy2017","Laguna2017","Waddell2017","Waddell2016","Lombardi2016","OldDairy2016","Younger2016","Scott2016","Laguna2016")
FSToutliers_95 <- FST_pairs > apply(FST_all, 1, quantile, probs = 0.95, na.rm = T)
write.table(FSToutliers_95, "update-FST_outliers_95_v1.tsv", col.names = TRUE, row.names = FALSE, quote = FALSE)

#FST-Overlap-AllYears
#done in terminal directly, here I'm gathering outliers shared at least twice (vtc=2)
awk -vtc=2 'NR==1{next};
             NR==2{for(i=2;i<=NF;i++){t[i]=0}};
             {for(i=2;i<=NF;i++){if($i=="TRUE"){t[i]++}}}
             END{
                 for(j in t) {
                     if(t[j]>=tc) {
                         print(j, t[j])
                     }
                 }
             }' update-FST_outliers_95_v1.tsv > update-outlier_overlap_2up.tsv


# Get outliers that overlap across N (e.g. 2 or more) population comparisons ----------
par_2up <- read.table("update-outlier_overlap_2up.tsv")
head(par_2up)
par_2up_index <- par_2up$V1

chromosome <- FST_all_info$V1[par_2up_index] # chromosome position for outlier
snp_pos <- FST_all_info$V2[par_2up_index] # SNP position for outlier
num_est_outlier <- par_2up$V2 # number of comparisons that the SNP is an outlier in

fst_outlier_2up <- as.data.frame(cbind(chromosome, snp_pos, num_est_outlier))
head(fst_outlier_2up)
dim(fst_outlier_2up)
write.table(fst_outlier_2up,"update-fst_outlier_2up.tsv", col.names = F, row.names = F,quote = F)


#THIS MUST BE RUN IN TERMINAL NOT R!!!
awk '{print $1, $2}' update-fst_outlier_2up.tsv > update-fst_outlier_2up_pos.tsv

#######ANNOTATION + FUNCTION STUFF ###########
#Downlaod Annotation
#version 5 ensemble Annotation
wget https://stickleback.genetics.uga.edu/downloadData/v5_assembly/stickleback_v5_ensembl_genes.gff3.gz

#prepare annotation file for next steps
gunzip(its-name)
grep "gene" stickleback_v5_ensembl_genes.gff3 > gene-stickleback_v5_ensembl_genes.gff3
grep "protein" gene-stickleback_v5_ensembl_genes.gff3 > protein-gene-stickleback_v5_ensembl_genes.gff3 #These files are the exact same!!!

#Now I need to look for matches. I am interested in the temporally repeated sites (shared witin an estuary across years). All this data is stored in v6counts-FST-shared-within-estuaries-all-merged-cutoff-added.tsv
#I need to prepare this file by returning the chrom. naming nomencalture to the chr. format
#I also want to split up the sites by number of times shared across space. Counts of 2 are temporally but NOT spatially shared. Counts of 3-11 are temporally and somewhat spatially shared, and counts of 12 are temporally and fully spatially shared (in all 6 estuaries over all 2 years)
#Done in R interactive (4.2.1)

#1. 
#Prep of the file:
setwd("/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/masked/AllYears-WithinEstuary/graphs-shared-outliers")
input_file<- "v6counts-FST-shared-within-estuaries-all-merged-cutoff-added.tsv"
df <- read.table(input_file, header = FALSE, row.names = NULL, sep = "\t", stringsAsFactors = FALSE)
df <- data.frame(do.call("rbind", strsplit(as.character(df$V1), " ")))
chr_mapping <- c(
    "1" = "chrI", "2" = "chrII", "3" = "chrIII", "4" = "chrIV", "5" = "chrV",
    "6" = "chrVI", "7" = "chrVII", "8" = "chrVIII", "9" = "chrIX", "10" = "chrX",
    "11" = "chrXI", "12" = "chrXII", "13" = "chrXIII", "14" = "chrXIV", "15" = "chrXV",
    "16" = "chrXVI", "17" = "chrXVII", "18" = "chrXVIII", "19" = "chrXIX", "20" = "chrXX",
    "21" = "chrXXI"
)


df$X1 <- ifelse(df$X1 %in% names(chr_mapping), chr_mapping[df$X1], df$X1)
# Split the data frame based on the values in column 3 but only include columns 1 + 2 (position data)
temp_no_spatial <- subset(df, X3 == 2, select = c("X1", "X2"))
temp_some_spatial <- subset(df, X3 %in% c(4, 6, 8, 10), select = c("X1", "X2"))
temp_full_spatial <- subset(df, X3 == 12, select = c("X1", "X2"))

# Write each subset to a separate output file
write.table(temp_no_spatial, "temporal-no-spatial-positions-for-GO.tsv", sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
write.table(temp_some_spatial, "temporal-some-spatial-positions-for-GO.tsv", sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
write.table(temp_full_spatial, "temporal-full-spatial-positions-for-GO.tsv", sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
#Then copied all 3 GO files to a new directory (annotation)


#2. Submit the following (as genehits,sh)
while read -r id pos
do
awk -v id=$id -v pos=$pos -f gene_hits.awk protein-gene-stickleback_v5_ensembl_genes.gff3
done < temporal-full-spatial-positions-for-GO.tsv> temporal-full-spatial-positions-for-GO-genehits.tsv

while read -r id pos
do
awk -v id=$id -v pos=$pos -f gene_hits.awk protein-gene-stickleback_v5_ensembl_genes.gff3
done < temporal-no-spatial-positions-for-GO.tsv> temporal-no-spatial-positions-for-GO-genehits.tsv

while read -r id pos
do
awk -v id=$id -v pos=$pos -f gene_hits.awk protein-gene-stickleback_v5_ensembl_genes.gff3
done < temporal-some-spatial-positions-for-GO.tsv> temporal-some-spatial-positions-for-GO-genehits.tsv

# genehits.awk was written by Alan??
#It is a separate script
        function within_range(val, lower, upper, proximity) {
            # you can specify the "proximity" as required
            return val > lower - proximity && val < upper + proximity
        }

        BEGIN {
            OFS="\t"
        }

        $1 == id && within_range(pos, $4, $5, 0) {
            name = gensub(/.*Name=([^\t]*).*/, "\\1", 1)
            if (name ~ /[^[:space:]]+/)
                print id, pos, name
        }

#3BIOMART
#####BACK TO ORIGINAL_ DOWNLOADING R packages######
#https://www.bioconductor.org/packages/release/bioc/html/biomaRt.html
#BIOMART- i followed the instructions for R4.3. Opened it in R 4.3 because 4.2.1 didn't work!!!
                            #installation- did this:
                            if (!require("BiocManager", quietly = TRUE))
                                install.packages("BiocManager")

                            BiocManager::install("biomaRt")

                            #For TOPGO (for 4.3 not 4.2.1, as above):
                            if (!require("BiocManager", quietly = TRUE))
                                install.packages("BiocManager")

                            BiocManager::install("topGO")
library(biomaRt)

#MANHATTEN PLOTS OF FST###
#this script is for the "simple" graphs with all points of each estuary!!
#Where you make the dataset needed
library(tidyr)
library(dplyr)
library(stringr)
setwd("/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/masked/AllYears-WithinEstuary")
input_file_name <- "AllYearsWithinestuary-REALmasked-autosome-chrom.fst"
output_file_name <- "output-man.tsv"
output_rename_file <- "rename-output-man.tsv"

df <- read.table(input_file_name, header = FALSE, sep = "\t", stringsAsFactors = FALSE)

tidy_data <- df %>%
  pivot_longer(cols = 6:17, names_to = "key", values_to = "value") %>%
  separate(value, into = c("sub_key", "sub_value"), sep = "=") %>%
  select(-c(3:6))


write.table(tidy_data, output_file_name, sep = "\t", col.names = FALSE, row.names = FALSE, quote = FALSE)

df <- read.table(output_file_name, header = FALSE, sep = "\t", stringsAsFactors = FALSE)

chr_mapping <- c(
  "chrI" = 1, "chrII" = 2, "chrIII" = 3, "chrIV" = 4, "chrV" = 5,
  "chrVI" = 6, "chrVII" = 7, "chrVIII" = 8, "chrIX" = 9, "chrX" = 10,
  "chrXI" = 11, "chrXII" = 12, "chrXIII" = 13, "chrXIV" = 14, "chrXV" = 15,
  "chrXVI" = 16, "chrXVII" = 17, "chrXVIII" = 18, "chrXIX" = 19, "chrXX" = 20,
  "chrXXI" = 21
)

df$V1 <- ifelse(df$V1 %in% names(chr_mapping), chr_mapping[df$V1], df$V1)

rename_mapping <- c(
  "1:11" = "Younger2017",
  "2:7" = "Scott2017",
  "3:9" = "Lombardi2017",
  "4:8" = "OldDairy2017",
  "5:10" = "Laguna2017",
  "6:12" = "Waddell2017",
  "13:19" = "Waddell2016",
  "14:16" = "Lombardi2016",
  "15:18" = "OldDairy2016",
  "17:24" = "Younger2016",
  "20:21" = "Scott2016",
  "22:23" = "Laguna2016"
)

df$V3 <- ifelse(df$V3 %in% names(rename_mapping), rename_mapping[df$V3], df$V3)

write.table(df, output_rename_file, sep = "\t", col.names = FALSE, row.names = FALSE, quote = FALSE)
#NOW GRAPH### 
#Note that each pop is a separate panel, grouped by years. Only the individual panels have the cutoff lines (not grouped)
#Adapted from https://danielroelfs.com/blog/how-i-create-manhattan-plots-using-ggplot/
library(ggplot2)
library(dplyr)

data <- read.table("rename-output-man.tsv", header = FALSE, col.names = c('chromosome', 'position', 'population', 'fst_value'))
data$Population_Group <- ifelse(grepl("2016", data$population, ignore.case = TRUE), "2016", "2017")
cutoff_data <- read.table("cutoff_values.csv", header = TRUE, sep = ",")
create_population_manhplot <- function(subdata, population, cutoff_data) {
  data_cum <- subdata %>%
    group_by(chromosome) %>%
    summarise(max_bp = max(position)) %>%
    mutate(bp_add = lag(cumsum(max_bp), default = 0)) %>%
    select(chromosome, bp_add)
 subdata <- subdata %>%
    inner_join(data_cum, by = "chromosome") %>%
    mutate(bp_cum = position + bp_add)

  axis_set <- subdata %>%
    group_by(chromosome) %>%
    summarise(center = mean(bp_cum))

manhplot <- ggplot(subdata, aes(x = bp_cum, y = fst_value, group = population, color = factor(chromosome))) +
    geom_point(alpha = 0.75) +
    scale_x_continuous(
      label = axis_set$chromosome,
      breaks = axis_set$center
    ) +
facet_wrap(~population, scales = "fixed", ncol = 1) +
    scale_y_continuous(limits = c(0, 1)) +
    scale_color_manual(values = rep(
      c("#276FBF", "#183059"),
      unique(length(axis_set$chromosome))
    )) +
    labs(title = paste("Update Manhattan Plot -", population), x = "Genomic Position", y = "FST Value") +
    theme_minimal() +
    theme(
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      axis.title.y = element_text(),
      axis.text.x = element_text(angle = 60, size = 8, vjust = 0.5),
      plot.background = element_rect(fill = "white"),
      panel.background = element_rect(fill = "white"),
      legend.position = "none"
    )

cutoff_value <- cutoff_data$CutoffValue[cutoff_data$PopulationPair == population]

manhplot <- manhplot +
    geom_hline(yintercept = cutoff_value, linetype = "dashed", color = "red", alpha = 0.7) +
    annotate("text", x = max(subdata$bp_cum), y = cutoff_value,
             label = paste("Cutoff: ", round(cutoff_value, 4)),
             hjust = 1, vjust = -0.5, color = "red")

  return(manhplot)
}

for (year in unique(data$Population_Group)) {
  year_plot <- create_population_manhplot(subset(data, Population_Group == year), year, cutoff_data)
  ggsave(paste0("Manhattan_", year, ".png"), year_plot)

  panels <- unique(subset(data, Population_Group == year)$population)
  for (panel in panels) {
    panel_data <- subset(data, Population_Group == year & population == panel)
    panel_plot <- create_population_manhplot(panel_data, panel, cutoff_data)
    ggsave(paste0("Manhattan_", year, "_", panel, ".png"), panel_plot)
  }
}

#To calculate the cutoff values for each population- 
#this script was written by me in Chat GPT and was added after the outliers were calculated. 
#Ran for all datasets in R interactive 
### UODATED SCRIPT TO ALSO CALCULATE MIN AND MAX OUTLIERS (min should = cutoff)
library(data.table)

setwd("/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/masked/AllYears-WithinEstuary")

FST_all <- fread("AllYearsWithinestuary-REALmasked-autosome-chrom.fst", header = FALSE, data.table = FALSE)

FST_all_info <- FST_all[, 1:5]
FST_all <- FST_all[, 6:17]

names_all <- sapply(FST_all, function(i) unique(sub('=.*', '', i)))

FST_all <- data.frame(lapply(FST_all, function(i) sub('.*=', '', i)))
names(FST_all) <- names_all
FST_all <- apply(as.matrix(FST_all), 2, as.numeric)
FST_all <- data.frame(t(FST_all))

FST_pairs <- as.data.frame(FST_all)
row.names(FST_pairs) <- c("Younger2017", "Scott2017", "Lombardi2017", "OldDairy2017", "Laguna2017", "Waddell2017", "Waddell2016", "Lombardi2016", "OldDairy2016", "Younger2016", "Scott2016", "Laguna2016")

find_outliers <- function(values, threshold) {
  return(values > threshold)
}

calculate_outliers <- function(population_name, fst_values) {
  threshold <- quantile(fst_values, probs = 0.95, na.rm = TRUE)
  outliers <- find_outliers(fst_values, threshold)
  min_outlier <- min(fst_values[outliers], na.rm = TRUE)
  max_outlier <- max(fst_values[outliers], na.rm = TRUE)
  
  return(c(Population = population_name, CutoffValue = threshold, MinOutlier = min_outlier, MaxOutlier = max_outlier))
}

result_df <- data.frame(Population = character(), CutoffValue = numeric(), MinOutlier = numeric(), MaxOutlier = numeric(), stringsAsFactors = FALSE)

for (population_name in rownames(FST_pairs)) {
  fst_values <- FST_pairs[population_name, , drop = FALSE]
  result_row <- calculate_outliers(population_name, fst_values)
  result_df <- rbind(result_df, result_row)
}

print(result_df)

write.csv(result_df, "outliers_summary.csv", row.names = FALSE)






            # I think this part will be junk, but tbd once the analysis above completes
            # Assuming FST_all and FST_pairs are already loaded and processed
            library(data.table)
            getwd()
            setwd("/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/masked/AllYears-WithinEstuary")
            FST_all<- fread("AllYearsWithinestuary-REALmasked-autosome-chrom.fst", header = F, data.table = F)
            dim(FST_all)
            FST_all_info <- FST_all[,1:5]
            FST_all <- FST_all[,6:17]
            names_all <- sapply(FST_all, function(i)unique(sub('=.*', '', i)))
            FST_all <- data.frame(lapply(FST_all, function(i)sub('.*=', '', i)))
            names(FST_all) <- names_all
            FST_all <- apply(as.matrix(FST_all), 2, as.numeric)
            FST_all <- data.frame(t(FST_all))
            FST_pairs <- FST_all[,]
            FST_pairs <- as.data.frame(FST_pairs)
            row.names(FST_pairs)<- c("Younger2017","Scott2017","Lombardi2017","OldDairy2017","Laguna2017","Waddell2017","Waddell2016","Lombardi2016","OldDairy2016","Younger2016","Scott2016","Laguna2016")
            FSToutliers_95 <- FST_pairs > apply(FST_all, 1, quantile, probs = 0.95, na.rm = T)
            FST_cutoff_values <- apply(FST_all, 1, quantile, probs = 0.95, na.rm = TRUE)
            FST_cutoff_df <- data.frame(PopulationPair = rownames(FST_all), CutoffValue = FST_cutoff_values)
            write.csv(FST_cutoff_df, "cutoff_values.csv", row.names = FALSE)


#My own personal script to count the number of times an outlier is in a column 
#(i.e. the number of outliers that are not shared, or shared 2-6 times):

awk -vtc=1 'NR==1{next};
             NR==2{for(i=2;i<=NF;i++){t[i]=0}};
             {for(i=2;i<=NF;i++){if($i=="TRUE"){t[i]++}}}
             END{
                 for(j in t) {
                     if(t[j]>=tc) {
                         print(j, t[j])
                     }
                 }
             }' update-FST_outliers_95_v1.tsv > update-non-overlap-outlier.tsv

#Version to check Alan's cutoff values
library(data.table)
setwd("/home/mary2018/projects/rrg-barrett/alan_mk_SC_stick")
FST_all <- fread("estuary_chr-all_snps_informative.fst", header = FALSE, data.table = FALSE)
FST_all_info <- FST_all[, 1:4]
FST_all <- FST_all[, 5:10]
names_all <- sapply(FST_all, function(i) unique(sub('=.*', '', i)))
FST_all <- data.frame(lapply(FST_all, function(i) sub('.*=', '', i)))
names(FST_all) <- names_all
FST_all <- apply(as.matrix(FST_all), 2, as.numeric)
FST_all <- data.frame(t(FST_all))
FST_pairs <- as.data.frame(FST_all)
row.names(FST_pairs) <- c( "Waddell?2016","Lombardi?2016",
                            "OldDairy?2016","Younger?2016",
                            "Scott?2016","Laguna?2016")
find_outliers <- function(values, threshold) {
  return(values > threshold)
}

calculate_outliers <- function(population_name, fst_values) {
  threshold <- quantile(fst_values, probs = 0.95, na.rm = TRUE)
  outliers <- find_outliers(fst_values, threshold)
  min_outlier <- min(fst_values[outliers], na.rm = TRUE)
  max_outlier <- max(fst_values[outliers], na.rm = TRUE)
 
  return(c(Population = population_name, CutoffValue = threshold, MinOutlier = min_outlier, MaxOutlier = max_outlier))
}

result_df <- data.frame(Population = character(), CutoffValue = numeric(), MinOutlier = numeric(), MaxOutlier = numeric(), stringsAsFactors = FALSE)

for (population_name in rownames(FST_pairs)) {
  fst_values <- FST_pairs[population_name, , drop = FALSE]
  result_row <- calculate_outliers(population_name, fst_values)
  result_df <- rbind(result_df, result_row)
}
write.csv(result_df, "outliers_summaryALAN.csv", row.names = FALSE)

#Count 1-6 (non-overlap)
#Written with chat 
input_file=update-non-overlap-outlier.tsv
output_file=update-nonoverlap_count.txt
for value in 1 2 3 4 5 6 7 8 9 10 11 12; do
count=$(awk -v val="$value" '$2 == val {count++} END {print count}' "$input_file")
echo "Count of $value in column 2: $count" >> "$output_file"
done


#####GENERAL INFO#########

#COPY fst outlier to graham because of beluga shutdown
*use ssh -A mary2018@graham.alliancecan.ca

#!/bin/bash
#SBATCH -J copy-fst-to-graham
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 4
#SBATCH --mem=60g
#SBATCH -t 08:10
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o  copy-fst-to-graham%j.o
#SBATCH -e copy-fst-to-graham%j.e
#SBATCH --account=def-barrett
scp REALmasked-autosome-chrom.fst mary2018@beluga.computecanada.ca:/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis

#https://docs.alliancecan.ca/wiki/Transferring_data#Between_resources

#Info on grep https://informatics.fas.harvard.edu/short-introduction-to-grep.html

##COPYING important files to graham as a backup!! Will copy the original files for 2016 + 2017, the masked and unmasked .sync files, and the fst output file for the masked version
#!/bin/bash
#SBATCH -J copy-fst-to-graham
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 4
#SBATCH --mem=120g
#SBATCH -t 24:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o  copy-fst-to-graham%j.o
#SBATCH -e copy-fst-to-graham%j.e
#SBATCH --account=def-barrett

scp HI.4271.00* mary2018@beluga.computecanada.ca:/home/mary2018/projects/rrg-barrett/mary2018/2016_Samples_RAW_Data/UncompressedRawData
scp Pool_* mary2018@beluga.computecanada.ca:/home/mary2018/projects/rrg-barrett/mary2018/2017_Samples_RAW-Data/RawData/FinalCopiedFASTQ
scp unmasked-autosome-chrom.sync mary2018@beluga.computecanada.ca:/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/unmasked
scp masked-autosome-chrom.sync mary2018@beluga.computecanada.ca:/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis/masked
scp REALmasked-autosome-chrom.fst mary2018@beluga.computecanada.ca:/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/analysis

#General guidelines
head #to see top 3 lines
cd .. #to go to previous level
cd #to change directory
rm #to remove
squeue --job jobnumber #to see job status
sq #shows all my jobs 
seff jobnumer #to see the usage of the job unpon completion (i.e. cpu etc)
wc -l #line count 

sacct -j 41496473 --format=jobid,jobname,reqcpus,reqmem,averss,maxrss,elapsed,state%20,exitcode --unit=M #get more info on the jub submitted (replace the number above with the job number) #For more info - https://hpc.nmsu.edu/discovery/slurm/job-management/

module avail #Tells you the versions of all available packagess

#How to loop FASTP through multiple files https://github.com/OpenGene/fastp/issues/106
#Useful videos https://www.youtube.com/watch?v=iutnoFqOee0
#How to check our usage on CC sshare -l -A rrg-barrett_cpu -a
    #you want LevelFS for the whole rrg-barrett to be above 1 - that means we have higher priority

head -n 1 SCMulti-filtered.mpileup | nano -  #how to see head from a file (n= 1 row) in nano (so that the line format is preserved)
head -n 30 finalSCmulti.mpileup | cut -f 1-4 | nano -      #same as above but only see the first 4 columns


#How to downlaod code from github (get the html code form the share on git page)
git clone html.address.etc


#Making an mpileup for wing (only 100 lines of original file)
#!/bin/bash
#SBATCH -J copy
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 1
#SBATCH -t 0:30:0
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=mk.hickox@mail.mcgill.ca
#SBATCH -o copy1%j.o
#SBATCH -e copy1%j.e
#SBATCH --account=rrg-barrett
cp SCMulti-MKattempt2Filtered.mpileup 

/home/mary2018/projects/rrg-barrett/mary2018/MultiYear/unfiltered
##OTHER##
TEST SCRIPTS:
/home/mary2018/scratch

/home/mary2018/.local/bin/popoolation2_1201


#GitHub
git init
git add -A 
nano .gitignore #if you want certain files to be ifnored, make a file with either the directory you want ignored or the file names 
git commit -m "initial commit"

#to check if local repository and remote repository in github worked properly, you can do ssh -T git@github.com
git remote add origin thenTheSSHPathFromGitHub
#Then we push it from the local 
git push -u -f origin markduplicateswithmatecigar

git pull #any updates on github are pulled back to this version

#How to make sandboxes (to get your own programs on your account)

#info on filtering

418393380 mk filtered version (Attempt 2) (9.6%), indels from that 15147521 (3.6%)
416425403 SC filtered (Antoine version, filtered with grep)
462920658 original unfilterede file 
348570 wing filtered version
5176185 perl synv file after 3 hours
418393380 line count 2hopefully.synv
lines in the output of masked/filtered file (should be 418393380-15147521=403245859)= 308605962 (26.24% of the filtered lines removed)
    #This is because the indels have start and end spots. This means that a given indel most likely removes multiple lines of the filtered file.

Wing version SCMulti-MyVersionfiltered.mpileup 
